## Cosmic Web - Synthesis

### Table of Contents

1. [Results](#1-results)
2. [One call per tool](#2-one-call-per-tool)
3. [Memory traffic](#3-memory-traffic)
4. [Our dataset](#4-our-dataset)
5. [Programming models](#5-programming-models)

---

### 1. Results

Each notebook saved its results under `timings/`. We compare the best version of each tool. The table and the sweep also list the versions they improved on.

In [ ]:
import numpy as np
import cupy as cp

import cosmic_core as cw

STAGES = ['cupy_views', 'jax', 'cccl', 'numba', 'cppjit_cuda']   # the best version of each tool
EXTRAS = ['cupy', 'jax_bincount', 'cppjit_cub']                   # the versions they improved on
LABELS = {'cupy': 'CuPy reference', 'cupy_views': 'CuPy views', 'jax': 'JAX slabs', 'jax_bincount': 'JAX bincount',
          'cccl': 'cuda.compute', 'numba': 'numba-cuda', 'cppjit_cub': 'CppJIT CUB', 'cppjit_cuda': 'CppJIT CUDA'}

by_stage = cw.load_stages(STAGES, EXTRAS)
N, K = by_stage['cupy']['n'], by_stage['cupy']['k']
cw.stage_table(by_stage, LABELS)

### 2. One call per tool

Does any tool leave the GPU idle while the CPU works?

In [ ]:
edges = cw.bin_edges(K)
reference = cw.chi_reference(N)
curves = {LABELS[s]: cw.ecc_from_hist(by_stage[s]['hist']) for s in STAGES if s in by_stage and 'hist' in by_stage[s]}
cw.plot_curves_with_delta(edges, curves, reference, title=f'χ(ν) at N = {N}^3 from every tool')

traced = [s for s in STAGES if s in by_stage and 'trace' in by_stage[s] and 'launches' in by_stage[s]]
cw.plot_timelines({LABELS[s]: by_stage[s]['trace'] for s in traced},
                  title='one call per tool: GPU operations on a shared time axis')

parts = {LABELS[s]: cw.wall_breakdown(by_stage[s]) for s in traced}
cw.plot_breakdown(parts, title='where one call spends its wall time')

- Time outside kernels and copies is small in every bar, so no tool leaves the GPU idle for long.

### 3. Memory traffic

How close does each tool get to its estimate? Each dashed line is the throughput a tool would reach if it moved its estimated bytes at the GPU's copy bandwidth.

In [ ]:
gbs = cw.copy_gbs()
swept = [s for s in by_stage if 'sweep' in by_stage[s]]
cw.sweep_series(by_stage, {s: LABELS[s] for s in swept})
cw.plot_traffic({LABELS[s]: by_stage[s]['sweep'] for s in swept}, {LABELS[s]: s for s in swept}, gbs,
                title=f'throughput against N; dashed: memory-traffic estimate at {gbs:.0f} GB/s')

first = [s for s in by_stage if 'cold_s' in by_stage[s]]
cw.plot_first_call([LABELS[s] for s in first], [by_stage[s]['cold_s'] for s in first],
                   [by_stage[s]['median_s'] for s in first], 'first call against the median')

- At the smallest size every tool is far from its estimate: launches and copies set the time.
- At the largest size CuPy reaches its estimate, with and without views. JAX, cuda.compute and the two kernels stay below theirs.

### 4. Our dataset

Let's draw the curve of our dataset against the Gaussian expectation.

In [ ]:
nu, lam, _ = cw.quijote_field()
q = cw.quantise(cp.asarray(nu), K)
hist = cp.asnumpy(cw.ecc_hist_cupy(q, K))
assert hist.sum() == 0, hist.sum()
chi = cw.ecc_from_hist(hist)
pred = cw.chi_gaussian(edges, lam, q.size)
_, zeros = cw.plot_overlay(edges, chi, pred, title=f'χ(ν) of our dataset at {q.shape[0]}^3', label='our dataset')
print(f'zero crossings: our dataset {np.round(zeros, 2).tolist()}, '
      f'the Gaussian expectation {np.round(cw.zero_crossings(edges, pred), 2).tolist()}')

- Our dataset is not a Gaussian field, so its curve departs from the formula.

### 5. Programming models

| tool | provides | we write | optimization |
|---|---|---|---|
| **CuPy** | one kernel per array expression | array expressions | views and preallocated buffers in place of shifted copies |
| **JAX** | tracing, XLA fusion, one executable | a function with static arguments | the fused chain, then slabs against scatter contention |
| **cuda.compute** | CUB algorithms with iterators compiled into their kernels | iterators and the algorithm call | iterators in place of stored birth arrays |
| **numba-cuda** | CUDA kernels compiled from Python | a kernel in Python and its launch | a single pass with a shared-memory histogram |
| **CppJIT** | C++ and CUDA compiled in the session | CUB's call, then a CUDA C++ kernel and its launch | a single kernel for the stencil and the histogram |

Takeaways:

- Every tool produced the same histogram, bin for bin, so the timings compare the same work.
- The first call of every tool includes its compile, build or load cost, paid once per session.
- Different programming models have their own areas of optimization. With enough profiling, bottlenecks can be identified and mitigated.
- Pick the tool that suits your problem best, based on your understanding of your data and the algorithm you want to run.
- Start with existing abstractions and drop into the lower-level tools that let you control your data and compute model only when the gains are clear.